# Lesson 4: Command Runner

Add `run_command` so the agent can execute shell commands — run tests, check git status, install packages, etc.

```mermaid
graph LR
    A[User Input] --> B[LLM]
    B -->|tool_call| C[list_files]
    B -->|tool_call| D[read_file]
    B -->|tool_call| E[run_command]
    C & D & E -->|result| B
    B -->|text| F[Response]
    F --> A
```

In [ ]:
%pip install openai-agents --upgrade --quiet

In [ ]:
import os
import subprocess
from getpass import getpass

from agents import Agent, Runner, function_tool

MODEL = "gpt-5.4-mini"

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

In [ ]:
@function_tool
def read_file(path: str) -> str:
    """Read the contents of a file at the given path."""
    try:
        with open(path, "r") as f:
            content = f.read()
        if len(content) > 10000:
            content = content[:10000] + "\n... (truncated)"
        return content
    except FileNotFoundError:
        return f"Error: File not found: {path}"
    except Exception as e:
        return f"Error reading file: {e}"


@function_tool
def list_files(path: str = ".") -> str:
    """List all files and directories at the given path. Directories end with /."""
    try:
        entries = []
        for entry in sorted(os.listdir(path)):
            full = os.path.join(path, entry)
            if os.path.isdir(full):
                entries.append(f"{entry}/")
            else:
                entries.append(entry)
        return "\n".join(entries) if entries else "(empty directory)"
    except FileNotFoundError:
        return f"Error: Directory not found: {path}"
    except Exception as e:
        return f"Error listing files: {e}"

## Defining the `run_command` Tool

This is the most powerful tool — it lets the agent run arbitrary shell commands. We add a timeout and output limit for safety.

In [ ]:
@function_tool
def run_command(command: str) -> str:
    """Run a shell command and return its output. Use this for running tests, checking git status, or any system command."""
    try:
        result = subprocess.run(
            command,
            shell=True,
            capture_output=True,
            text=True,
            timeout=30,
        )
        output = result.stdout
        if result.stderr:
            output += f"\nSTDERR:\n{result.stderr}"
        if result.returncode != 0:
            output += f"\n(exit code: {result.returncode})"
        if len(output) > 10000:
            output = output[:10000] + "\n... (truncated)"
        return output if output.strip() else "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Command timed out after 30 seconds"
    except Exception as e:
        return f"Error running command: {e}"

In [ ]:
agent = Agent(
    name="Command Runner",
    instructions="""You are a coding assistant that can explore files and run shell commands.
Use list_files and read_file to understand the codebase.
Use run_command to execute shell commands like running tests, checking versions, or inspecting the system.""",
    model=MODEL,
    tools=[read_file, list_files, run_command],
)

In [ ]:
result = await Runner.run(agent, "What version of Python is installed? And what's the current git branch?")
print(result.final_output)

## Summary

- Added `run_command` with timeout and output limits for safety.
- The agent can now explore files *and* interact with the system.
- Next lesson: add `edit_file` so the agent can modify code.